### Initialisation

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../outputs/data/train_data.csv')

### Data Preparation

In [3]:
# Select features (same as Random Forest for fair comparison)
key_features = [
    'EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_1',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
    'DAYS_EMPLOYED_YEARS', 'DAYS_BIRTH_YEARS',
    'AMT_CREDIT', 'AMT_INCOME_TOTAL',
    'CNT_FAM_MEMBERS',
    'EMPLOYMENT_TO_AGE_RATIO',
]

# Add other numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
additional_cols = [c for c in numeric_cols if c not in key_features + ['TARGET', 'SK_ID_CURR']]
key_features.extend(additional_cols[:30])

# Create feature matrix
X = df[key_features].copy()
y = df['TARGET'].copy()

print(f"\nUsing {X.shape[1]} features")
print(f"Total samples: {X.shape[0]:,}")
print(f"Default rate: {y.mean():.2%}")

# Handle missing values
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0)



Using 41 features
Total samples: 307,511
Default rate: 8.07%


### Train-Test Split

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\nTraining: {X_train.shape[0]:,} samples")
print(f"Validation: {X_val.shape[0]:,} samples")
print(f"Default rate in training: {y_train.mean():.2%}")

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


Training: 246,008 samples
Validation: 61,503 samples
Default rate in training: 8.07%
scale_pos_weight: 11.39


### LightGBM with Hyperparameter Tuning

In [5]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [6]:
import lightgbm as lgb

# Base LightGBM model
lgb_base = lgb.LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    metric='auc'
)

# Hyperparameter grid (LightGBM has different params than XGBoost)
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700],
    'max_depth': [3, 5, 7, 9, 11, -1],  # -1 = no limit
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.07, 0.1],
    'num_leaves': [15, 31, 63, 127, 255],  # Key LightGBM param
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [5, 10, 20, 30, 50],  # Equivalent to min_child_weight
    'reg_alpha': [0, 0.1, 0.5, 1.0, 2.0],  # L1 regularization
    'reg_lambda': [0, 0.1, 0.5, 1.0, 2.0],  # L2 regularization
}

In [7]:
print("Searching hyperparameters (5-10 minutes)...")

random_search = RandomizedSearchCV(
    lgb_base,
    param_distributions=param_dist,
    n_iter=40,  # more niter than XGBoost (onyl 30)
    scoring='roc_auc',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print(f"\nBest ROC-AUC from CV: {random_search.best_score_:.4f}")
print(f"Best parameters: {random_search.best_params_}")

# Train final model with best parameters
best_lgb = random_search.best_estimator_

Searching hyperparameters (5-10 minutes)...
Fitting 3 folds for each of 40 candidates, totalling 120 fits

Best ROC-AUC from CV: 0.7467
Best parameters: {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 2.0, 'num_leaves': 15, 'n_estimators': 200, 'min_child_samples': 50, 'max_depth': 11, 'learning_rate': 0.07, 'colsample_bytree': 0.9}


### Evaluation

In [8]:
y_pred_proba = best_lgb.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred_proba)

print(f"\n{'='*50}")
print(f"LIGHTGBM RESULTS")
print(f"{'='*50}")
print(f"ROC-AUC: {auc:.4f}")

# Comparison with previous models
print(f"\nMODEL COMPARISON:")
print(f"  Random Forest (baseline):   0.701")
print(f"  Random Forest (optimized):  0.737")
print(f"  XGBoost (tuned):            0.753")
print(f"  LightGBM (tuned):           {auc:.4f}")

# Determine best model
models_auc = {
    'Random Forest (baseline)': 0.701,
    'Random Forest (optimized)': 0.737,
    'XGBoost': 0.753,
    'LightGBM': auc
}
best_model = max(models_auc, key=models_auc.get)
best_score = models_auc[best_model]

print(f"\n BEST MODEL: {best_model} with ROC-AUC {best_score:.4f}")

if auc > 0.753:
    print(f"   LightGBM improves over XGBoost by +{(auc - 0.753)*100:.2f} percentage points")
elif auc > 0.76:
    print(f"   LightGBM achieves {auc:.4f} — strong result!")
else:
    print(f"   LightGBM similar to XGBoost. Both are valid.")


LIGHTGBM RESULTS
ROC-AUC: 0.7519

MODEL COMPARISON:
  Random Forest (baseline):   0.701
  Random Forest (optimized):  0.737
  XGBoost (tuned):            0.753
  LightGBM (tuned):           0.7519

 BEST MODEL: XGBoost with ROC-AUC 0.7530
   LightGBM similar to XGBoost. Both are valid.


### Find Optimal Threshold

In [9]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)

# Find threshold that captures 80% of defaults
target_recall = 0.80
closest_idx = (np.abs(recalls - target_recall)).argmin()
business_threshold = thresholds[closest_idx] if closest_idx < len(thresholds) else 0.5

# Find threshold that maximizes F1 score
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
best_f1_idx = np.argmax(f1_scores)
f1_threshold = thresholds[best_f1_idx] if best_f1_idx < len(thresholds) else 0.5

print(f"Threshold to capture 80% of defaults: {business_threshold:.3f}")
print(f"Threshold for best F1 score: {f1_threshold:.3f} (F1 = {f1_scores[best_f1_idx]:.3f})")

# Apply business threshold
y_pred_business = (y_pred_proba > business_threshold).astype(int)

Threshold to capture 80% of defaults: 0.406
Threshold for best F1 score: 0.664 (F1 = 0.306)


### Confusion Matrix and Business Metrics

In [10]:
tn, fp, fn, tp = confusion_matrix(y_val, y_pred_business).ravel()

print(f"  True Positives (correctly identified defaults): {tp:,}")
print(f"  False Positives (incorrectly flagged as risky): {fp:,}")
print(f"  False Negatives (missed defaults): {fn:,}")
print(f"  True Negatives (correctly approved): {tn:,}")

precision_business = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_business = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n  Precision: {precision_business:.2%} (of flagged, this many actually default)")
print(f"  Recall: {recall_business:.2%} (of defaults, this many caught)")
print(f"  Efficiency gain: {precision_business / y_val.mean():.1f}x better than random")

  True Positives (correctly identified defaults): 3,972
  False Positives (incorrectly flagged as risky): 25,457
  False Negatives (missed defaults): 993
  True Negatives (correctly approved): 31,081

  Precision: 13.50% (of flagged, this many actually default)
  Recall: 80.00% (of defaults, this many caught)
  Efficiency gain: 1.7x better than random


### Feature Importance

In [11]:
# Get feature importance
feature_importance_lgb = pd.DataFrame({
    'feature': X.columns,
    'importance': best_lgb.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
for i, row in feature_importance_lgb.head(10).iterrows():
    print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")


Top 10 Most Important Features:
  3. EXT_SOURCE_1: 240.0000
  2. EXT_SOURCE_3: 221.0000
  1. EXT_SOURCE_2: 216.0000
  13. AMT_ANNUITY: 206.0000
  8. AMT_CREDIT: 204.0000
  14. AMT_GOODS_PRICE: 182.0000
  16. DAYS_BIRTH: 132.0000
  19. DAYS_ID_PUBLISH: 119.0000
  7. DAYS_BIRTH_YEARS: 107.0000
  11. EMPLOYMENT_TO_AGE_RATIO: 105.0000


### Comparison: LightGBM vs XGBoost 

In [12]:
# Load XGBoost feature importance if available
try:
    xgb_importance = pd.read_csv('../outputs/results/xgboost_feature_importance.csv')
    print("\nTop 5 Features Comparison:")
    print(f"{'Rank':<5} {'LightGBM':<25} {'XGBoost':<25}")
    print("-" * 55)
    for i in range(5):
        lgb_feat = feature_importance_lgb.iloc[i]['feature'] if i < len(feature_importance_lgb) else '-'
        xgb_feat = xgb_importance.iloc[i]['feature'] if i < len(xgb_importance) else '-'
        print(f"{i+1:<5} {lgb_feat:<25} {xgb_feat:<25}")
except:
    print("  XGBoost feature importance not found — skipping comparison")


Top 5 Features Comparison:
Rank  LightGBM                  XGBoost                  
-------------------------------------------------------
1     EXT_SOURCE_1              EXT_SOURCE_2             
2     EXT_SOURCE_3              EXT_SOURCE_3             
3     EXT_SOURCE_2              EXT_SOURCE_1             
4     AMT_ANNUITY               DAYS_EMPLOYED_YEARS      
5     AMT_CREDIT                FLAG_EMP_PHONE           


### Save Results

In [13]:
import joblib

# Save LightGBM model
joblib.dump(best_lgb, '../outputs/models/lightgbm_model.pkl')
print("> LightGBM model saved to outputs/models/lightgbm_model.pkl")

# Save feature importance
feature_importance_lgb.to_csv('../outputs/results/lightgbm_feature_importance.csv', index=False)
print("> LightGBM feature importance saved")

# Save predictions
predictions_df = X_val.copy()
predictions_df['actual_default'] = y_val
predictions_df['default_probability'] = y_pred_proba
predictions_df['risk_flag'] = y_pred_business
predictions_df.to_csv('../outputs/results/lightgbm_predictions.csv', index=False)
print("> Predictions saved to outputs/results/lightgbm_predictions.csv")

> LightGBM model saved to outputs/models/lightgbm_model.pkl
> LightGBM feature importance saved
> Predictions saved to outputs/results/lightgbm_predictions.csv


# Final Summary

In [16]:
print("\n" + "=" * 60)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│                     MODEL PERFORMANCE COMPARISON                │
├─────────────────────────────────────────────────────────────────┤
│  Model                          ROC-AUC    Improvement          │
├─────────────────────────────────────────────────────────────────┤
│  Random Forest (baseline)       0.701      —                    │
│  Random Forest (optimized)      0.737      +3.6%                │
│  XGBoost (tuned)                0.753      +5.2%                │
│  LightGBM (tuned)               {auc:.4f}      +{(auc - 0.701)*100:.1f}%               │
├─────────────────────────────────────────────────────────────────┤
│  BEST MODEL: {best_model:<25} {best_score:.4f}                   │
└─────────────────────────────────────────────────────────────────┘

BUSINESS METRICS (LightGBM at 80% recall threshold):
├── Threshold: {business_threshold:.3f}
├── Precision: {precision_business:.2%}
├── Recall: {recall_business:.2%}
└── Efficiency: {precision_business / y_val.mean():.1f}x better than random

TOP 5 FEATURES (LightGBM):
{chr(10).join([f"  {i+1}. {row['feature']}" for i, row in feature_importance_lgb.head(5).iterrows()])}
""")




FINAL MODEL COMPARISON SUMMARY

┌─────────────────────────────────────────────────────────────────┐
│                     MODEL PERFORMANCE COMPARISON                │
├─────────────────────────────────────────────────────────────────┤
│  Model                          ROC-AUC    Improvement          │
├─────────────────────────────────────────────────────────────────┤
│  Random Forest (baseline)       0.701      —                    │
│  Random Forest (optimized)      0.737      +3.6%                │
│  XGBoost (tuned)                0.753      +5.2%                │
│  LightGBM (tuned)               0.7519      +5.1%               │
├─────────────────────────────────────────────────────────────────┤
│  BEST MODEL: XGBoost                   0.7530                   │
└─────────────────────────────────────────────────────────────────┘

BUSINESS METRICS (LightGBM at 80% recall threshold):
├── Threshold: 0.406
├── Precision: 13.50%
├── Recall: 80.00%
└── Efficiency: 1.7x better than ra